In [1]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from pathlib import Path
from tqdm import tqdm

# ─── CONFIG ───────────────────────────────────────────────────────────────────
CSV_PATH    = r"C:\Users\Tanishq\Downloads\final_dataset_467.csv"
OUTPUT_DIR  = r"C:\JupyterNotebook\MRI_PET\Data"
TARGET_SIZE = (128, 128, 128)
# ──────────────────────────────────────────────────────────────────────────────

# Create output folders
mri_out = Path(OUTPUT_DIR) / "MRI"
pet_out = Path(OUTPUT_DIR) / "PET"
mri_out.mkdir(parents=True, exist_ok=True)
pet_out.mkdir(parents=True, exist_ok=True)


def load_dicom_series(folder_path):
    """Load DICOM series from folder and return SimpleITK image."""
    reader = sitk.ImageSeriesReader()
    dicom_files = reader.GetGDCMSeriesFileNames(str(folder_path))
    if len(dicom_files) == 0:
        raise ValueError(f"No DICOM files found in {folder_path}")
    reader.SetFileNames(dicom_files)
    return reader.Execute()


def skull_strip_mri(image):
    """Simple skull stripping using Otsu thresholding + largest component."""
    otsu_filter = sitk.OtsuThresholdImageFilter()
    otsu_filter.SetInsideValue(0)
    otsu_filter.SetOutsideValue(1)
    mask = otsu_filter.Execute(image)

    cc        = sitk.ConnectedComponent(mask)
    sorted_cc = sitk.RelabelComponent(cc, sortByObjectSize=True)
    largest   = sitk.BinaryThreshold(sorted_cc, lowerThreshold=1, upperThreshold=1)

    return sitk.Mask(image, largest)


def register_pet_to_mri(mri_image, pet_image):
    """
    Register PET to MRI space using rigid registration.
    This ensures same slice in MRI and PET shows same brain region.
    """
    # Cast both to float for registration
    mri_float = sitk.Cast(mri_image, sitk.sitkFloat32)
    pet_float = sitk.Cast(pet_image, sitk.sitkFloat32)

    # Initialize registration
    registration = sitk.ImageRegistrationMethod()

    # Metric: Mattes Mutual Information (best for MRI-PET since they look different)
    registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    registration.SetMetricSamplingStrategy(registration.RANDOM)
    registration.SetMetricSamplingPercentage(0.1)

    # Interpolator
    registration.SetInterpolator(sitk.sitkLinear)

    # Optimizer
    registration.SetOptimizerAsGradientDescent(
        learningRate=1.0,
        numberOfIterations=100,
        convergenceMinimumValue=1e-6,
        convergenceWindowSize=10
    )
    registration.SetOptimizerScalesFromPhysicalShift()

    # Multi-resolution strategy (speeds up registration)
    registration.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
    registration.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
    registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    # Initial transform: align centers of both images
    initial_transform = sitk.CenteredTransformInitializer(
        mri_float,
        pet_float,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )
    registration.SetInitialTransform(initial_transform, inPlace=False)

    # Run registration
    final_transform = registration.Execute(mri_float, pet_float)

    # Apply transform: resample PET into MRI space
    pet_registered = sitk.Resample(
        pet_image,
        mri_image,           # use MRI as reference space
        final_transform,
        sitk.sitkLinear,
        0.0,
        pet_image.GetPixelID()
    )

    return pet_registered


def resize_volume(image, target_size):
    """Resize a SimpleITK image to target size."""
    original_size    = image.GetSize()
    original_spacing = image.GetSpacing()

    new_spacing = [
        original_spacing[i] * (original_size[i] / target_size[i])
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(target_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0)

    return resampler.Execute(image)


def normalize_mri(array):
    """Z-score normalization for MRI (ignores background zeros)."""
    nonzero = array[array > 0]
    if len(nonzero) == 0:
        return array
    mean = nonzero.mean()
    std  = nonzero.std()
    if std == 0:
        return array
    normalized = (array - mean) / std
    normalized[array == 0] = 0
    return normalized


def normalize_pet(array):
    """Min-max normalization for PET → range [0, 1]."""
    min_val = array.min()
    max_val = array.max()
    if max_val == min_val:
        return array
    return (array - min_val) / (max_val - min_val)


# ─── MAIN LOOP ────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Processing {len(df)} pairs...\n")

success = []
failed  = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    subject_id   = str(row["Subject_ID"])
    mri_image_id = str(int(row["MRI_Image_ID"]))
    pet_image_id = str(int(row["PET_Image_ID"]))
    label        = row["Class_Label"]
    numeric      = row["Numeric_Label"]
    mri_path     = Path(row["MRI_Path"])
    pet_path     = Path(row["PET_Path"])

    mri_npy_name = f"{subject_id}_I{mri_image_id}.npy"
    pet_npy_name = f"{subject_id}_I{pet_image_id}.npy"
    mri_npy_path = mri_out / mri_npy_name
    pet_npy_path = pet_out / pet_npy_name

    # Skip if already processed (resume support)
    if mri_npy_path.exists() and pet_npy_path.exists():
        success.append({
            "Subject_ID"   : subject_id,
            "Class_Label"  : label,
            "Numeric_Label": numeric,
            "MRI_File"     : mri_npy_name,
            "PET_File"     : pet_npy_name,
        })
        continue

    try:
        # ── Load ─────────────────────────────────────────
        mri_img = load_dicom_series(mri_path)
        pet_img = load_dicom_series(pet_path)

        # ── Skull strip MRI ───────────────────────────────
        mri_img = skull_strip_mri(mri_img)

        # ── Register PET → MRI space ─────────────────────
        pet_img = register_pet_to_mri(mri_img, pet_img)

        # ── Resize both to 128x128x128 ───────────────────
        mri_img = resize_volume(mri_img, TARGET_SIZE)
        pet_img = resize_volume(pet_img, TARGET_SIZE)

        # ── Convert to numpy ─────────────────────────────
        mri_arr = sitk.GetArrayFromImage(mri_img).astype(np.float32)
        pet_arr = sitk.GetArrayFromImage(pet_img).astype(np.float32)

        # ── Normalize ────────────────────────────────────
        mri_arr = normalize_mri(mri_arr)
        pet_arr = normalize_pet(pet_arr)

        # ── Save ─────────────────────────────────────────
        np.save(mri_npy_path, mri_arr)
        np.save(pet_npy_path, pet_arr)

        success.append({
            "Subject_ID"   : subject_id,
            "Class_Label"  : label,
            "Numeric_Label": numeric,
            "MRI_File"     : mri_npy_name,
            "PET_File"     : pet_npy_name,
        })

    except Exception as e:
        failed.append({
            "Subject_ID"  : subject_id,
            "MRI_Image_ID": mri_image_id,
            "PET_Image_ID": pet_image_id,
            "Error"       : str(e),
        })

# ─── SAVE LABELS CSV ──────────────────────────────────────────────────────────
success_df = pd.DataFrame(success)
success_df.to_csv(Path(OUTPUT_DIR) / "labels.csv", index=False)

if failed:
    failed_df = pd.DataFrame(failed)
    failed_df.to_csv(Path(OUTPUT_DIR) / "failed.csv", index=False)

# ─── SUMMARY ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"✅ Successfully processed : {len(success)}")
print(f"❌ Failed                 : {len(failed)}")
if len(success) > 0:
    print("\nClass distribution:")
    print(success_df["Class_Label"].value_counts())
print(f"\n📄 Labels saved to : {OUTPUT_DIR}\\labels.csv")
if failed:
    print(f"⚠️  Failed log     : {OUTPUT_DIR}\\failed.csv")

Processing 467 pairs...



100%|██████████████████████████████████████████████████████████████████████████████| 467/467 [1:48:44<00:00, 13.97s/it]


PREPROCESSING SUMMARY
✅ Successfully processed : 465
❌ Failed                 : 2

Class distribution:
Class_Label
MCI    206
CN     196
AD      63
Name: count, dtype: int64

📄 Labels saved to : C:\JupyterNotebook\MRI_PET\Data\labels.csv
⚠️  Failed log     : C:\JupyterNotebook\MRI_PET\Data\failed.csv


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.5.1+cu121
True
NVIDIA GeForce RTX 4060 Laptop GPU
